<a href="https://colab.research.google.com/github/JinwaOssData/OssData01/blob/main/std001.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2026-2-중간-오픈소스기반데이터분석

공공데이터포털(https://data.go.kr)에서 "한국농수산식품유통공사_일별 도,소매 가격정보 조회

https://www.data.go.kr/data/15156057/openapi.do

1-2 API를 호출하는 파이썬 코드를 작성하고, 2015년부터 2024년까지 서울 가락도매시장의 배추(품종: 봄/여름(고랭지)/가을/월동) 가격 데이터를 수집하는 프로그램을 작성하시오. 파이썬 코드와 API 호출 성공을 확인할 수 있는 실행 결과를 캡처하여 첨부하시오. (7점)

In [ ]:
import requests

## 데이터 수집 url 및 api key 설정
url = 'https://apis.data.go.kr/B552845/perDay/price'
api_key = '30500eeedd0fdc424318fd7b05b8107397f71b09e06be0713833a5e1aee01f9b'

params = {
    'serviceKey': api_key,
    'returnType': 'json',
    'numOfRows': '200',
    'pageNo': '1',
    'cond[exmn_ymd::GTE]': '20150101',
    'cond[exmn_ymd::LTE]': '20241231',
    'cond[se_cd::EQ]': '02',
    'cond[ctgry_cd::EQ]': '200',
    'cond[item_cd::EQ]': '211',
    'cond[grd_cd::EQ]': '04',
    'cond[sgg_cd::EQ]': '1101',
    'cond[mrkt_cd::EQ]': '0110211',
    'vrty_cd': ['01','02','03','06'] ## 품종코드 (01 봄, 02 여름(고랭지), 03 가을, 06 월동)
}

## 데이터 수집
response = requests.get(url, params=params)

## 호출 성공/실패 출력
if response.status_code == 200:
    print('API 호출 성공')
else:
    print('API 호출 실패')


API 호출 성공


문 2-1. 수집한 JSON 형태의 데이터를 pandas DataFrame으로 변환하고, 데이터의 기본 정보를 출력하는 파이썬 코드와 실행 결과를 첨부하시오. (4점)

In [32]:
import requests
import pandas as pd

## 데이터 수집 url 및 api key 설정
url = 'https://apis.data.go.kr/B552845/perDay/price'
api_key = '30500eeedd0fdc424318fd7b05b8107397f71b09e06be0713833a5e1aee01f9b'

params = {
    'serviceKey': api_key, 'returnType': 'json', 'numOfRows': '200', 'pageNo': '1',
    'cond[exmn_ymd::GTE]': '20150101', 'cond[exmn_ymd::LTE]': '20241231',
    'cond[se_cd::EQ]': '02', 'cond[ctgry_cd::EQ]': '200', 'cond[item_cd::EQ]': '211', 'cond[grd_cd::EQ]': '04',
    'cond[sgg_cd::EQ]': '1101', 'cond[mrkt_cd::EQ]': '0110211',
    'vrty_cd': ['01','02','03','06'] ## 품종코드 (01 봄, 02 여름(고랭지), 03 가을, 06 월동)
}

## 데이터 수집
response = requests.get(url, params=params)
data = response.json()['response']['body']['items']['item']

df = pd.DataFrame(data)
print(df.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   ctgry_cd          200 non-null    object
 1   ctgry_nm          200 non-null    object
 2   exmn_dd_cnvs_prc  200 non-null    object
 3   exmn_dd_prc       200 non-null    object
 4   exmn_ymd          200 non-null    object
 5   grd_cd            200 non-null    object
 6   grd_nm            200 non-null    object
 7   item_cd           200 non-null    object
 8   item_nm           200 non-null    object
 9   mrkt_cd           200 non-null    object
 10  mrkt_nm           200 non-null    object
 11  orgnl_reg_dt      200 non-null    object
 12  se_cd             200 non-null    object
 13  se_nm             200 non-null    object
 14  sgg_cd            200 non-null    object
 15  sgg_nm            200 non-null    object
 16  unit              200 non-null    object
 17  unit_sz         

문 2-2. 연도별, 계절별 분석을 위해 조사일자(exmn_ymd) 컬럼을 활용하여 연도(year)와 계절(season) 컬럼을 추가하는 전처리 코드를 작성하고, 변환 결과를 확인할 수 있는 출력 결과를 첨부하시오. (4점)
※ 계절 구분: 봄(3-5월), 여름(6-8월), 가을(9-11월), 겨울(12-2월)

In [27]:
import requests
import json
import pandas as pd

## 데이터 수집 url 및 api key 설정
url = 'https://apis.data.go.kr/B552845/perDay/price'
api_key = '30500eeedd0fdc424318fd7b05b8107397f71b09e06be0713833a5e1aee01f9b'

params = {
    'serviceKey': api_key,
    'returnType': 'json',
    'numOfRows': '200',
    'pageNo': '1',
    'cond[exmn_ymd::GTE]': '20150101',
    'cond[exmn_ymd::LTE]': '20241231',
    'cond[se_cd::EQ]': '02',
    'cond[ctgry_cd::EQ]': '200',
    'cond[item_cd::EQ]': '211',
    'cond[grd_cd::EQ]': '04',
    'cond[sgg_cd::EQ]': '1101',
    'cond[mrkt_cd::EQ]': '0110211',
    'vrty_cd': ['01','02','03','06'] ## 품종코드 (01 봄, 02 여름(고랭지), 03 가을, 06 월동)
}

## 데이터 수집
response = requests.get(url, params=params)

## 데이터 파싱
data = response.json()['response']['body']['items']['item']

## pandas DataFrame 으로 변환
df = pd.DataFrame(data)

## pd 출력
df['exmn_ymd_dt'] = pd.to_datetime(df['exmn_ymd'].astype(str), format='%Y%m%d', errors='coerce')
#df['year'] = df['exmn_ymd'].dt.year

df

,ctgry_cd,ctgry_nm,exmn_dd_cnvs_prc,exmn_dd_prc,exmn_ymd,grd_cd,grd_nm,item_cd,item_nm,mrkt_cd,...,orgnl_reg_dt,se_cd,se_nm,sgg_cd,sgg_nm,unit,unit_sz,vrty_cd,vrty_nm,exmn_ymd_dt
0,200,채소류,4000,4000,20150102,04,상품,211,배추,0110211,...,2015-01-02T13:57:56Z,02,중도매,1101,서울,kg(그물망 3포기),10,03,가을,2015-01-02
1,200,채소류,4000,4000,20150105,04,상품,211,배추,0110211,...,2015-01-05T14:20:57Z,02,중도매,1101,서울,kg(그물망 3포기),10,03,가을,2015-01-05
2,200,채소류,4000,4000,20150106,04,상품,211,배추,0110211,...,2015-01-06T13:49:28Z,02,중도매,1101,서울,kg(그물망 3포기),10,03,가을,2015-01-06
3,200,채소류,4000,4000,20150107,04,상품,211,배추,0110211,...,2015-01-07T13:49:11Z,02,중도매,1101,서울,kg(그물망 3포기),10,06,월동,2015-01-07
4,200,채소류,4000,4000,20150108,04,상품,211,배추,0110211,...,2015-01-08T13:50:57Z,02,중도매,1101,서울,kg(그물망 3포기),10,06,월동,2015-01-08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,200,채소류,7000,7000,20150918,04,상품,211,배추,0110211,...,2015-09-18T14:11:30Z,02,중도매,1101,서울,kg(그물망 3포기),10,02,여름(고랭지),2015-09-18
196,200,채소류,6000,6000,20150921,04,상품,211,배추,0110211,...,2015-09-21T13:59:45Z,02,중도매,1101,서울,kg(그물망 3포기),10,02,여름(고랭지),2015-09-21
197,200,채소류,6500,6500,20150922,04,상품,211,배추,0110211,...,2015-09-22T14:38:24Z,02,중도매,1101,서울,kg(그물망 3포기),10,02,여름(고랭지),2015-09-22
198,200,채소류,7000,7000,20150923,04,상품,211,배추,0110211,...,2015-09-23T13:54:04Z,02,중도매,1101,서울,kg(그물망 3포기),10,02,여름(고랭지),2015-09-23
